# Stage A3 — Linear stitching

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
The strong test. For each matched layer pair, fits **one ridge-regression
matrix** (closed form — the models themselves are never trained) mapping
GPT-2's representation into Pythia's, and measures held-out R².

The transformation class is the whole game:
- *Any* nonlinear translator → vacuous (can map anything to anything)
- *Identity* (no transform) → trivially fails (permutation symmetry)
- **Linear** → strong enough to undo rotations/permutations/scalings,
  too weak to create information that is not already there.

## Success criterion
Held-out R² > 0.7 in the middle layers (edges are tokenizer-specific and
always lower), and far above the random-baseline curve.


In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# Functions
def stitch_r2(X, Y, alpha=1.0):
    """Fit Y ~ X @ W with ridge; return held-out R^2 (variance-weighted)."""
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25,
                                          random_state=0)
    # standardize inputs for stable ridge
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    reg = Ridge(alpha=alpha).fit(Xtr, Ytr)
    return r2_score(Yte, reg.predict(Xte),
                    multioutput="variance_weighted")


def matched_indices(nA, nB):
    """Match layers proportionally (models may differ in depth)."""
    return [(i, round(i * (nB - 1) / (nA - 1))) for i in range(nA)]


def main():
    data = np.load(str(DATA_DIR / "activations.npz"))
    A, B, R = data["A_layers"], data["B_layers"], data["R_layers"]

    pairs = matched_indices(A.shape[0], B.shape[0])
    r2_trained, r2_random = [], []

    for i, j in pairs:
        rt = stitch_r2(A[i], B[j])
        rr = stitch_r2(A[i], R[j])
        r2_trained.append(rt)
        r2_random.append(rr)
        print(f"GPT-2 L{i:2d} -> Pythia L{j:2d}:  "
              f"trained R2={rt:.3f}   random R2={rr:.3f}")

    layers = [i for i, _ in pairs]
    plt.figure(figsize=(9, 5))
    plt.plot(layers, r2_trained, "o-", label="A -> B (trained)")
    plt.plot(layers, r2_random, "s--", label="A -> random baseline")
    plt.axhline(0.7, color="gray", ls=":", label="success threshold")
    plt.xlabel("GPT-2 layer")
    plt.ylabel("Held-out R² of a single linear map")
    plt.title("Linear stitching: are the spaces isomorphic?")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "stitching_r2.png"), dpi=150)

    print(f"\nMean R2 trained: {np.mean(r2_trained):.3f}")
    print(f"Mean R2 random:  {np.mean(r2_random):.3f}")
    print("Saved stitching_r2.png")
    print("Hypothesis supported if trained curve sits high (>0.7 mid-layers) "
          "and far above the random baseline.")

In [ ]:
# Run the stitching (requires activations.npz from stage A1)
main()

## A3-bis — Direction-symmetry check (reverse stitching)

The main run fits GPT-2 -> Pythia. That direction is a convention: the
hypothesis ("isomorphic up to a linear map") is symmetric, so if it holds,
the reverse fit Pythia -> GPT-2 should score comparably. This cell runs
both directions per layer and compares.

**Why the two directions need not be *identical*:** ridge is not symmetric
in practice — the inverted matrix is built from the source side only, the
minimized error lives on the target side, and the two spaces differ in
anisotropy, so predicting *into* a more concentrated space is a slightly
different task than predicting out of one. The test is therefore about
*comparability*, not equality.

**Pre-registered reading:**
| Outcome | Interpretation |
|---|---|
| mean gap < 0.05, curves parallel | symmetric isomorphism — strengthens the main claim |
| gap 0.05–0.15 | mild asymmetry, consistent with anisotropy differences; report the number |
| gap > 0.15, or opposite depth trends | one space carries information the other lacks — a finding in itself, investigate before publishing |

Runtime: same as the main run (~1 min); requires `activations.npz`.

In [ ]:
# A3-bis: reverse-direction stitching
data = np.load(str(DATA_DIR / "activations.npz"))
A, B = data["A_layers"], data["B_layers"]
pairs = matched_indices(A.shape[0], B.shape[0])

fwd, rev = [], []
for i, j in pairs:
    f = stitch_r2(A[i], B[j])          # GPT-2 -> Pythia (main direction)
    r = stitch_r2(B[j], A[i])          # Pythia -> GPT-2 (reverse)
    fwd.append(f); rev.append(r)
    print(f"L{i:2d}:  A->B R2={f:.3f}   B->A R2={r:.3f}   "
          f"gap={f-r:+.3f}")

fwd, rev = np.array(fwd), np.array(rev)
gap = float(np.abs(fwd - rev).mean())
corr = float(np.corrcoef(fwd, rev)[0, 1])
print(f"\nmean A->B: {fwd.mean():.3f}   mean B->A: {rev.mean():.3f}")
print(f"mean |gap|: {gap:.3f}    depth-profile correlation: {corr:.3f}")

layers = [i for i, _ in pairs]
plt.figure(figsize=(9, 5))
plt.plot(layers, fwd, "o-", label="GPT-2 -> Pythia (main)")
plt.plot(layers, rev, "s-", label="Pythia -> GPT-2 (reverse)")
plt.axhline(0.7, color="gray", ls=":", label="success threshold")
plt.xlabel("layer"); plt.ylabel("Held-out R² of a single linear map")
plt.title("Direction symmetry: stitching both ways")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(str(DATA_DIR / "stitching_symmetry.png"), dpi=150)
print("Saved stitching_symmetry.png")

if gap < 0.05 and corr > 0.9:
    print("VERDICT: symmetric — supports isomorphism in the strong, "
          "two-way sense")
elif gap <= 0.15:
    print("VERDICT: mildly asymmetric — consistent with anisotropy "
          "differences between the spaces; report the number")
else:
    print("VERDICT: substantially asymmetric — one space predicts the "
          "other better than vice versa; investigate (anisotropy spectra, "
          "per-direction variance) before drawing conclusions")

## A3-ter — Confirmation tests for the direction asymmetry

Two measured explanations for the A3-bis asymmetry, plus the
direct anisotropy measurement. Self-contained; requires only
`activations.npz`. Test 1 re-derives the sentences with A1's
exact loader (rows must align). Runtime: ~2 minutes.

In [ ]:
# ==========================================================
# A3-ter — self-contained. Paste as ONE cell after A3-bis.
# ==========================================================
import os
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import AutoTokenizer

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
MODEL_A  = "gpt2"
MODEL_B  = "EleutherAI/pythia-160m"
MAX_LEN  = 64                      # must match what A1 used

data = np.load(str(DATA_DIR / "activations.npz"))
A, B = data["A_layers"], data["B_layers"]
N = A.shape[1]
print(f"loaded A {A.shape}, B {B.shape}")


# ---------- helpers ----------
def reload_sentences(n, min_chars=100):
    """Mirror of A1's load_sentences: same configs, filters, dedup, order.
    activations.npz does not store the text, so it must be re-derived, and
    row i here must be the same passage as row i of A/B."""
    for config in ("wikitext-103-raw-v1", "wikitext-2-raw-v1"):
        try:
            ds = load_dataset("Salesforce/wikitext", config, split="train",
                              streaming=True)
            seen, out = set(), []
            for row in ds:
                t = row["text"].strip()
                if len(t) < min_chars or t.startswith("="):
                    continue
                if t in seen:
                    continue
                seen.add(t)
                out.append(t)
                if len(out) >= n:
                    break
            if len(out) >= n:
                print(f"corpus: {config}  ({len(out)} passages)")
                return out
            print(f"{config} yielded only {len(out)} - trying next")
        except Exception as e:
            print(f"{config} unavailable ({type(e).__name__}) - trying next")
    raise RuntimeError("could not re-derive the sentence list")


def length_r2(M, lengths):
    """Held-out R2 for predicting sequence length from the pooled vector."""
    Xtr, Xte, ytr, yte = train_test_split(M, lengths, test_size=0.25,
                                          random_state=0)
    return r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))


def stitch_r2_mo(X, Y, multioutput, alpha=1.0):
    """A3's stitch_r2 with a selectable multioutput mode."""
    Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25,
                                          random_state=0)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    reg = Ridge(alpha=alpha).fit(Xtr, Ytr)
    return r2_score(Yte, reg.predict(Xte), multioutput=multioutput)


# ---------- TEST 1: length signal at layer 0 ----------
sentences = reload_sentences(N)
assert len(sentences) == N, "row count mismatch - alignment broken"

tok_a = AutoTokenizer.from_pretrained(MODEL_A)
tok_b = AutoTokenizer.from_pretrained(MODEL_B)
len_a = np.array([min(len(tok_a.encode(s)), MAX_LEN) for s in sentences])
len_b = np.array([min(len(tok_b.encode(s)), MAX_LEN) for s in sentences])
print(f"\ntoken lengths: GPT-2 mean {len_a.mean():.1f}, "
      f"Pythia mean {len_b.mean():.1f}, cap {MAX_LEN}")
print(f"fraction hitting the cap: GPT-2 {(len_a == MAX_LEN).mean():.2f}, "
      f"Pythia {(len_b == MAX_LEN).mean():.2f}")

print("\nTEST 1 - length recoverable from the pooled layer-0 vector:")
ra = length_r2(A[0], len_a)
rb = length_r2(B[0], len_b)
print(f"  GPT-2  L0: R2 = {ra:.3f}")
print(f"  Pythia L0: R2 = {rb:.3f}")
print("  expectation: GPT-2 high (learned absolute positional embeddings "
      "are added at layer 0),")
print("               Pythia low (rotary encoding is applied inside "
      "attention, not at layer 0)")
print("  VERDICT:", "supports the positional explanation"
      if ra - rb > 0.15 else "does NOT support it - look elsewhere")

# ---------- TEST 2: variance weighting in the middle layers ----------
print("\nTEST 2 - does the middle-layer gap shrink without variance "
      "weighting?")
gaps = {}
for i in (4, 5, 6):
    for mo in ("variance_weighted", "uniform_average"):
        f = stitch_r2_mo(A[i], B[i], mo)
        r = stitch_r2_mo(B[i], A[i], mo)
        gaps.setdefault(mo, []).append(f - r)
        print(f"  L{i}  {mo:18s}  A->B {f:.3f}   B->A {r:.3f}   "
              f"gap {f - r:+.3f}")
vw = float(np.mean(np.abs(gaps["variance_weighted"])))
ua = float(np.mean(np.abs(gaps["uniform_average"])))
print(f"\n  mean |gap|: variance_weighted {vw:.3f} -> uniform_average "
      f"{ua:.3f}")
print("  VERDICT:", "variance weighting explains most of the asymmetry"
      if ua < vw * 0.6 else "the asymmetry survives - not a weighting "
      "artifact")

# ---------- OPTIONAL: anisotropy directly ----------
def eff_dim(M):
    ev = np.clip(np.linalg.eigvalsh(np.cov(M.T.astype(np.float64))), 0, None)
    return float(ev.sum() ** 2 / (ev ** 2).sum())

print("\nOPTIONAL - effective dimensionality (of 768; lower = more "
      "anisotropic):")
for i in (0, 4, 8, 12):
    print(f"  L{i:2d}:  GPT-2 {eff_dim(A[i]):6.1f}    "
          f"Pythia {eff_dim(B[i]):6.1f}")
